# Product Alternative Finder Model

## Goal
Find cheaper products similar to a given product using **TF-IDF + Cosine Similarity**

## How It Works
1. **Extract** product names from raw data
2. **Clean** text (remove special characters, convert to lowercase)
3. **Vectorize** using TF-IDF (convert text to numerical vectors)
4. **Search** for cheaper products with high similarity scores
5. **Rank** by similarity and show savings percentage

In [3]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer

## Step 1: Import Required Libraries

In [4]:
df = pd.read_csv('cleaned_data.csv')

## Step 2: Load & Explore Data
Load product data from CSV and check for duplicates

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.head()

,Product Name,Price,Type,Source,Product_Name_Clean,Price_Numeric
0,"Samsung Galaxy S26 5G (Black, 12GB RAM, 256GB ...","87,999",Mobile,Amazon,samsung galaxy s26 5g black 12gb ram 256gb sto...,87999.0
1,"Samsung Galaxy S26 Ultra 5G (Black, 12GB RAM, ...","1,59,999",Mobile,Amazon,samsung galaxy s26 ultra 5g black 12gb ram g51...,159999.0
2,"Redmi A4 5G (Sparkle Purple, 4GB RAM, 128GB St...","11,999",Mobile,Amazon,redmi a4 5g sparkle purple 4gb ram 128gb stora...,11999.0
3,"iQOO Z11x 5G (Prismatic Green, 6GB RAM, 128 GB...","18,998",Mobile,Amazon,iqoo z11x 5g prismatic green 6gb ram 128 gb st...,18998.0
4,"Samsung Galaxy M06 5G Mobile (Blazing Black, 6...","11,999",Mobile,Amazon,samsung galaxy m06 5g mobile blazing black 6gb...,11999.0


In [7]:
# Extract product name by:
# Step 1: Remove content inside parentheses
# Step 2: Split on comma or pipe, keep first part
# Step 3: Strip extra spaces

df['Product_Name_Extracted'] = df['Product Name'].apply(
    lambda x: re.split(r'[,|]', re.sub(r"\(.*?\)", "", x))[0].strip()
)

## Step 3: Extract Product Names
Remove noise from product names (parentheses, extra text) to get clean names

In [8]:
print("Full extracted name for row 1:")
print(repr(df['Product_Name_Extracted'].iloc[1]))
print("\nFirst 10 extracted product names:")
print(df['Product_Name_Extracted'].head(10).tolist())

Full extracted name for row 1:
'Samsung Galaxy S26 Ultra 5G  with Built-in Privacy Display'

First 10 extracted product names:
['Samsung Galaxy S26 5G', 'Samsung Galaxy S26 Ultra 5G  with Built-in Privacy Display', 'Redmi A4 5G', 'iQOO Z11x 5G', 'Samsung Galaxy M06 5G Mobile', 'Samsung Galaxy M07 Mobile', 'realme NARZO 90x 5G', 'Samsung Galaxy M17e 5G Mobile', 'Samsung Galaxy M17e 5G Mobile', 'iQOO Z10x 5G']


In [9]:
# Clean extracted product names for TF-IDF
def clean_text_for_tfidf(text):
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and numbers, keep only letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

df['Product_Name_Clean'] = df['Product_Name_Extracted'].apply(clean_text_for_tfidf)
print("Cleaned product names (first 10):")
print(df[['Product_Name_Extracted', 'Product_Name_Clean']].head(10))

Cleaned product names (first 10):
                              Product_Name_Extracted  \
0                              Samsung Galaxy S26 5G   
1  Samsung Galaxy S26 Ultra 5G  with Built-in Pri...   
2                                        Redmi A4 5G   
3                                       iQOO Z11x 5G   
4                       Samsung Galaxy M06 5G Mobile   
5                          Samsung Galaxy M07 Mobile   
6                                realme NARZO 90x 5G   
7                      Samsung Galaxy M17e 5G Mobile   
8                      Samsung Galaxy M17e 5G Mobile   
9                                       iQOO Z10x 5G   

                                  Product_Name_Clean  
0                                 samsung galaxy s g  
1  samsung galaxy s ultra g with builtin privacy ...  
2                                          redmi a g  
3                                          iqoo zx g  
4                          samsung galaxy m g mobile  
5                  

## Step 4: Clean Product Names
Convert to lowercase and remove special characters (only keep letters and spaces)

In [10]:
# Apply TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=50, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Product_Name_Clean'])

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"\nTop features (terms):")
print(tfidf_vectorizer.get_feature_names_out())

TF-IDF Matrix shape: (176, 50)

Top features (terms):
['acer' 'aspire' 'asus' 'core' 'dell' 'display' 'fhd' 'flipkart' 'galaxy'
 'gaming' 'gb' 'gen' 'hp' 'hz' 'ideapad' 'ih' 'intel' 'ips' 'iqoo'
 'laptop' 'lenovo' 'light' 'mobile' 'model' 'motorola' 'narzo' 'nord'
 'omnibook' 'oneplus' 'oppo' 'pavilion' 'plus' 'power' 'primebook'
 'privacy' 'pro' 'professional' 'ram' 'realme' 'redmi' 'samsung' 'slim'
 'smartchoice' 'supervooc' 'th' 'tuf' 'ultra' 'victus' 'vivobook' 'zx']


## Step 5: Convert Text to Numbers (TF-IDF Vectorization)
Transform product names into numerical vectors for similarity comparison
- TF-IDF = Term Frequency-Inverse Document Frequency
- Creates a 176×50 matrix (176 products, 50 features)

In [11]:
# ============================================================================
# MODEL: Find Cheaper Similar Products using TF-IDF + Cosine Similarity
# ============================================================================
from sklearn.metrics.pairwise import cosine_similarity

def find_cheaper_alternatives(product_name, top_n=5):
    """
    Find cheaper products similar to the input product.
    
    Parameters:
    -----------
    product_name : str
        The product name to search for (e.g., 'iQOO Z11x 5G')
    top_n : int
        Number of cheaper alternatives to return
    
    Returns:
    --------
    DataFrame with cheaper similar products
    """
    
    # STEP 1: Search for the product (case-insensitive)
    matching_products = df[df['Product_Name_Extracted'].str.contains(product_name, case=False, na=False)]
    if matching_products.empty:
        return f"Product '{product_name}' not found in dataset"
    
    # STEP 2: Get product index and its price
    product_idx = matching_products.index[0]
    original_price = df.loc[product_idx, 'Price_Numeric']
    
    # STEP 3: Convert product to numerical vector (TF-IDF representation)
    product_vector = tfidf_matrix[product_idx]
    
    # STEP 4: Calculate similarity with ALL products using Cosine Similarity
    # Score: 0 = completely different, 1 = identical
    similarities = cosine_similarity(product_vector, tfidf_matrix)[0]
    
    # STEP 5: Extract cheaper products
    cheaper_mask = df['Price_Numeric'] < original_price
    cheaper_indices = df[cheaper_mask].index.tolist()
    
    # STEP 6: Pair each cheaper product with its similarity score and sort
    cheaper_similarities = [(idx, similarities[idx]) for idx in cheaper_indices]
    cheaper_similarities.sort(key=lambda x: x[1], reverse=True)  # Highest similarity first
    
    # STEP 7: Take top N results
    top_cheaper = cheaper_similarities[:top_n]
    result_indices = [idx for idx, sim in top_cheaper]
    
    # STEP 8: Build result DataFrame with all useful info
    results = df.loc[result_indices, ['Product_Name_Extracted', 'Price_Numeric', 'Source', 'Type']].copy()
    results['Similarity_Score'] = [sim for idx, sim in top_cheaper]
    results['Original_Price'] = original_price
    results['Price_Difference'] = results['Original_Price'] - results['Price_Numeric']
    results['Savings_Percentage'] = (results['Price_Difference'] / results['Original_Price'] * 100).round(2)
    
    return results

# Test the model
test_product = 'HP Victus'
print(f"\n{'='*80}")
print(f"Finding cheaper alternatives for: {test_product}")
print(f"{'='*80}\n")
recommendations = find_cheaper_alternatives(test_product, top_n=5)
print(recommendations)


Finding cheaper alternatives for: HP Victus

    Product_Name_Extracted  Price_Numeric    Source    Type  Similarity_Score  \
42                   HP 15        42990.0    Amazon  Laptop          0.558594   
43                   HP 15        57990.0    Amazon  Laptop          0.558594   
130                  HP 15        42990.0  Flipkart  Laptop          0.558594   
131                  HP 15        57990.0  Flipkart  Laptop          0.558594   
52      HP Professional 14        41909.0    Amazon  Laptop          0.312028   

     Original_Price  Price_Difference  Savings_Percentage  
42          61990.0           19000.0               30.65  
43          61990.0            4000.0                6.45  
130         61990.0           19000.0               30.65  
131         61990.0            4000.0                6.45  
52          61990.0           20081.0               32.39  


## Step 6: Build the Model Function
Create a reusable function that:
1. Takes a product name as input
2. Finds matching products in the dataset
3. Calculates similarity using cosine distance
4. Returns cheaper alternatives ranked by similarity

In [12]:

# Test with different products
test_products = ['HP Victus', 'Motorola G57 Power 5G', 'POCO C71']

for product in test_products:
    print(f"\n{'='*80}")
    print(f"Cheaper alternatives for: {product}")
    print(f"{'='*80}\n")
    result = find_cheaper_alternatives(product, top_n=1)
    if isinstance(result, str):
        print(result)
    else:
        print(result[['Product_Name_Extracted', 'Price_Numeric', 'Savings_Percentage']])
    print()



Cheaper alternatives for: HP Victus

   Product_Name_Extracted  Price_Numeric  Savings_Percentage
42                  HP 15        42990.0               30.65


Cheaper alternatives for: Motorola G57 Power 5G

   Product_Name_Extracted  Price_Numeric  Savings_Percentage
16  Motorola G57 Power 5G        14920.0                1.84


Cheaper alternatives for: POCO C71

      Product_Name_Extracted  Price_Numeric  Savings_Percentage
5  Samsung Galaxy M07 Mobile         7999.0               19.99



## Step 7: Test with Sample Products
Test the model on real products to see results

In [13]:

# Model Summary
print("\n" + "="*80)
print("MODEL SUMMARY: Find Cheaper Product Alternatives")
print("="*80)
print("""
HOW IT WORKS:
1. Takes a product name as input (e.g., 'iQOO Z11x 5G')
2. Searches for the product in the dataset
3. Uses TF-IDF + Cosine Similarity to find similar products
4. Filters for products that are CHEAPER
5. Returns top N alternatives ranked by similarity

OUTPUT COLUMNS:
- Product_Name_Extracted: Name of the cheaper alternative
- Price_Numeric: Price of the alternative
- Similarity_Score: How similar it is (0-1, higher = more similar)
- Price_Difference: Amount saved in rupees
- Savings_Percentage: Percentage discount compared to original

USAGE:
  find_cheaper_alternatives('iQOO Z11x 5G', top_n=5)
  find_cheaper_alternatives('Samsung Galaxy S26 5G', top_n=3)
  find_cheaper_alternatives('Motorola G57 Power 5G', top_n=7)
""")
print("="*80)



MODEL SUMMARY: Find Cheaper Product Alternatives

HOW IT WORKS:
1. Takes a product name as input (e.g., 'iQOO Z11x 5G')
2. Searches for the product in the dataset
3. Uses TF-IDF + Cosine Similarity to find similar products
4. Filters for products that are CHEAPER
5. Returns top N alternatives ranked by similarity

OUTPUT COLUMNS:
- Product_Name_Extracted: Name of the cheaper alternative
- Price_Numeric: Price of the alternative
- Similarity_Score: How similar it is (0-1, higher = more similar)
- Price_Difference: Amount saved in rupees
- Savings_Percentage: Percentage discount compared to original

USAGE:
  find_cheaper_alternatives('iQOO Z11x 5G', top_n=5)
  find_cheaper_alternatives('Samsung Galaxy S26 5G', top_n=3)
  find_cheaper_alternatives('Motorola G57 Power 5G', top_n=7)



## Step 8: Model Summary & Usage
How to use this model in your own code

In [16]:
# Load the larger bigdata CSV
df_raw = pd.read_csv('amazon_flipkart_products_bigdata.csv')

print(f"Dataset shape: {df_raw.shape}")
print(f"\nColumn names: {df_raw.columns.tolist()}")
print(f"\nFirst few rows:")
print(df_raw.head(3))

Dataset shape: (3818, 5)

Column names: ['Product Name', 'Price (₹)', 'Category', 'Product Type', 'Platform']

First few rows:
            Product Name  Price (₹)     Category Product Type  Platform
0   OnePlus 12 5G - Gold      44455  Electronics   Smartphone    Amazon
1   OnePlus 12 5G - Gold      48101  Electronics   Smartphone  Flipkart
2  OnePlus 12 5G - White      47015  Electronics   Smartphone    Amazon


In [17]:
# Check data types and missing values
print("Data Info:")
print(df_raw.info())
print("\n Missing values:")
print(df_raw.isnull().sum())
print(f"\n Duplicates: {df_raw.duplicated().sum()}")

Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Product Name  3818 non-null   object
 1   Price (₹)     3818 non-null   int64 
 2   Category      3818 non-null   object
 3   Product Type  3818 non-null   object
 4   Platform      3818 non-null   object
dtypes: int64(1), object(4)
memory usage: 149.3+ KB
None

 Missing values:
Product Name    0
Price (₹)       0
Category        0
Product Type    0
Platform        0
dtype: int64

 Duplicates: 0


In [18]:
# ============================================================================
# DATA CLEANING FOR TF-IDF & COSINE SIMILARITY
# ============================================================================

# Step 1: Rename and standardize columns to match cleaned_data.csv format
df_clean = df_raw.copy()
df_clean.rename(columns={
    'Price (₹)': 'Price',
    'Product Type': 'Type',
    'Platform': 'Source'
}, inplace=True)

print("✓ Columns renamed")
print(f"  New columns: {df_clean.columns.tolist()}")

# Step 2: Extract numeric price (remove commas)
def extract_price_numeric(price):
    """Convert price string to float"""
    if pd.isna(price):
        return None
    price_str = str(price).replace(',', '').strip()
    try:
        return float(price_str)
    except:
        return None

df_clean['Price_Numeric'] = df_clean['Price'].apply(extract_price_numeric)
print(f"\n✓ Price_Numeric extracted")
print(f"  Sample prices: {df_clean['Price_Numeric'].head(3).tolist()}")

# Step 3: Extract product names (remove content in parentheses, split on comma/pipe)
df_clean['Product_Name_Extracted'] = df_clean['Product Name'].apply(
    lambda x: re.split(r'[,|]', re.sub(r"\(.*?\)", "", x))[0].strip()
)
print(f"\n✓ Product_Name_Extracted created")
print(f"  Sample extracted names:")
for i in range(3):
    print(f"    {df_clean['Product Name'].iloc[i]}")
    print(f"    → {df_clean['Product_Name_Extracted'].iloc[i]}")

# Step 4: Clean text for TF-IDF (lowercase, remove special chars)
def clean_text_for_tfidf(text):
    """Clean text for TF-IDF vectorization"""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join(text.split())
    return text

df_clean['Product_Name_Clean'] = df_clean['Product_Name_Extracted'].apply(clean_text_for_tfidf)
print(f"\n✓ Product_Name_Clean created (for TF-IDF)")
print(f"  Sample cleaned names:")
for i in range(3):
    print(f"    {df_clean['Product_Name_Extracted'].iloc[i]}")
    print(f"    → {df_clean['Product_Name_Clean'].iloc[i]}")

# Step 5: Keep only required columns (same format as cleaned_data.csv)
df_cleaned_final = df_clean[['Product_Name_Extracted', 'Price', 'Type', 'Source', 
                              'Product_Name_Clean', 'Price_Numeric']].copy()

# Rename for consistency with original dataset
df_cleaned_final.columns = ['Product_Name_Extracted', 'Price', 'Type', 'Source',
                             'Product_Name_Clean', 'Price_Numeric']

print(f"\n✓ Final cleaned dataset shape: {df_cleaned_final.shape}")
print(f"\nFinal dataset preview:")
print(df_cleaned_final.head(5))

✓ Columns renamed
  New columns: ['Product Name', 'Price', 'Category', 'Type', 'Source']

✓ Price_Numeric extracted
  Sample prices: [44455.0, 48101.0, 47015.0]

✓ Product_Name_Extracted created
  Sample extracted names:
    OnePlus 12 5G - Gold
    → OnePlus 12 5G - Gold
    OnePlus 12 5G - Gold
    → OnePlus 12 5G - Gold
    OnePlus 12 5G - White
    → OnePlus 12 5G - White

✓ Product_Name_Clean created (for TF-IDF)
  Sample cleaned names:
    OnePlus 12 5G - Gold
    → oneplus g gold
    OnePlus 12 5G - Gold
    → oneplus g gold
    OnePlus 12 5G - White
    → oneplus g white

✓ Final cleaned dataset shape: (3818, 6)

Final dataset preview:
  Product_Name_Extracted  Price        Type    Source Product_Name_Clean  \
0   OnePlus 12 5G - Gold  44455  Smartphone    Amazon     oneplus g gold   
1   OnePlus 12 5G - Gold  48101  Smartphone  Flipkart     oneplus g gold   
2  OnePlus 12 5G - White  47015  Smartphone    Amazon    oneplus g white   
3  OnePlus 12 5G - White  48418  Smartphone 

In [20]:
# ============================================================================
# VERIFY CLEANED DATA IS READY FOR TF-IDF & COSINE SIMILARITY
# ============================================================================

print("📊 CLEANED DATA READINESS CHECK\n")

# Check 1: TF-IDF compatibility
print("1️⃣  TF-IDF Compatibility:")
print(f"   ✓ All product names cleaned: {df_cleaned_final['Product_Name_Clean'].notna().all()}")
print(f"   ✓ Product names have content: {(df_cleaned_final['Product_Name_Clean'].str.len() > 0).all()}")
print(f"   ✓ Sample cleaned names: {df_cleaned_final['Product_Name_Clean'].head(3).tolist()}")

# Check 2: Cosine similarity compatibility  
print("\n2️⃣  Cosine Similarity Compatibility:")
print(f"   ✓ All prices numeric: {df_cleaned_final['Price_Numeric'].dtype == 'float64'}")
print(f"   ✓ No missing prices: {df_cleaned_final['Price_Numeric'].notna().all()}")
print(f"   ✓ Price statistics:")
print(f"     - Mean: ₹{df_cleaned_final['Price_Numeric'].mean():.2f}")
print(f"     - Median: ₹{df_cleaned_final['Price_Numeric'].median():.2f}")
print(f"     - Std Dev: ₹{df_cleaned_final['Price_Numeric'].std():.2f}")

# Check 3: Ready to use
print("\n3️⃣  Ready for Model:")
print(f"   ✓ Cleaned data file: {output_filename}")
print(f"   ✓ Total records: {len(df_cleaned_final)}")
print(f"   ✓ Can use for TF-IDF vectorization: YES ✅")
print(f"   ✓ Can use for cosine similarity: YES ✅")

print("\n" + "="*60)
print("📁 Next Step: Use cleaned_bigdata.csv in app.py or analysis")
print("="*60)

📊 CLEANED DATA READINESS CHECK

1️⃣  TF-IDF Compatibility:
   ✓ All product names cleaned: True
   ✓ Product names have content: True
   ✓ Sample cleaned names: ['oneplus g gold', 'oneplus g gold', 'oneplus g white']

2️⃣  Cosine Similarity Compatibility:
   ✓ All prices numeric: True
   ✓ No missing prices: True
   ✓ Price statistics:
     - Mean: ₹26380.31
     - Median: ₹9709.50
     - Std Dev: ₹42147.41

3️⃣  Ready for Model:
   ✓ Cleaned data file: cleaned_bigdata.csv
   ✓ Total records: 3818
   ✓ Can use for TF-IDF vectorization: YES ✅
   ✓ Can use for cosine similarity: YES ✅

📁 Next Step: Use cleaned_bigdata.csv in app.py or analysis


In [19]:
# Save the cleaned dataset
output_filename = 'cleaned_bigdata.csv'
df_cleaned_final.to_csv(output_filename, index=False)

print(f"✅ Cleaned dataset saved to: {output_filename}")
print(f"\nDataset Summary:")
print(f"  • Total products: {len(df_cleaned_final)}")
print(f"  • Product categories: {df_cleaned_final['Type'].nunique()}")
print(f"  • Sources: {df_cleaned_final['Source'].unique().tolist()}")
print(f"  • Price range: ₹{df_cleaned_final['Price_Numeric'].min():.0f} - ₹{df_cleaned_final['Price_Numeric'].max():.0f}")
print(f"  • Missing prices: {df_cleaned_final['Price_Numeric'].isnull().sum()}")
print(f"\nCategories breakdown:")
print(df_cleaned_final['Type'].value_counts())
print(f"\nSource breakdown:")
print(df_cleaned_final['Source'].value_counts())

✅ Cleaned dataset saved to: cleaned_bigdata.csv

Dataset Summary:
  • Total products: 3818
  • Product categories: 32
  • Sources: ['Amazon', 'Flipkart']
  • Price range: ₹1 - ₹361962
  • Missing prices: 0

Categories breakdown:
Type
Shoes              240
Smartphone         232
Headphones         232
Laptop             224
Television         202
Washing Machine    178
Refrigerator       162
Book               162
Jeans              153
Skincare           148
Jacket             146
Shirt              136
T-Shirt            129
Air Fryer          124
Tablet             116
Smartwatch         100
Camera              98
Pressure Cooker     95
Microwave           94
Kurta               88
Vacuum Cleaner      88
Water Purifier      84
Toothbrush          72
Shaving             71
Dishwasher          70
Drone               62
Soap                59
Book Set            58
Epilator            52
Sunscreen           52
Lipstick            50
Body Wash           41
Name: count, dtype: int64

Sou

In [24]:
# ============================================================================
# COMBINE BOTH DATASETS
# ============================================================================

# Standardize column names in original dataset
df_original_std = df_original.copy()
if 'Product Name' in df_original_std.columns:
    df_original_std.rename(columns={'Product Name': 'Product_Name_Extracted'}, inplace=True)

# Combine both dataframes using concatenation
df_combined = pd.concat([df_original_std, df_bigdata], ignore_index=True)

print("🔄 Combining datasets...")
print(f"\n  Original dataset: {len(df_original_std)} products")
print(f"  BigData dataset:  {len(df_bigdata)} products")
print(f"  Combined total:   {len(df_combined)} products")

# Remove duplicates (same product name and price)
df_combined_dedup = df_combined.drop_duplicates(
    subset=['Product_Name_Extracted', 'Price_Numeric'],
    keep='first'
).reset_index(drop=True)

print(f"\n✨ After removing duplicates: {len(df_combined_dedup)} products")
print(f"  Duplicates removed: {len(df_combined) - len(df_combined_dedup)}")

# Display sample of combined data
print(f"\n📊 Combined Dataset Preview:")
print(df_combined_dedup.head(10))

# Summary statistics
print(f"\n📈 Combined Dataset Statistics:")
print(f"  • Total products: {len(df_combined_dedup)}")
print(f"  • Product types: {df_combined_dedup['Type'].nunique()}")
print(f"  • Sources: {df_combined_dedup['Source'].unique().tolist()}")
print(f"  • Price range: ₹{df_combined_dedup['Price_Numeric'].min():.0f} - ₹{df_combined_dedup['Price_Numeric'].max():.0f}")
print(f"  • Average price: ₹{df_combined_dedup['Price_Numeric'].mean():.2f}")

🔄 Combining datasets...

  Original dataset: 176 products
  BigData dataset:  3818 products
  Combined total:   3994 products

✨ After removing duplicates: 3992 products
  Duplicates removed: 2

📊 Combined Dataset Preview:
                              Product_Name_Extracted     Price    Type  \
0  Samsung Galaxy S26 5G (Black, 12GB RAM, 256GB ...    87,999  Mobile   
1  Samsung Galaxy S26 Ultra 5G (Black, 12GB RAM, ...  1,59,999  Mobile   
2  Redmi A4 5G (Sparkle Purple, 4GB RAM, 128GB St...    11,999  Mobile   
3  iQOO Z11x 5G (Prismatic Green, 6GB RAM, 128 GB...    18,998  Mobile   
4  Samsung Galaxy M06 5G Mobile (Blazing Black, 6...    11,999  Mobile   
5  Samsung Galaxy M07 Mobile (Black, 4GB RAM, 64G...     7,999  Mobile   
6  realme NARZO 90x 5G (Marine Blue,4GB+128GB) |7...    12,999  Mobile   
7  Samsung Galaxy M17e 5G Mobile (Blitz Blue, 6GB...    15,499  Mobile   
8  Samsung Galaxy M17e 5G Mobile (Vibe Violet, 4G...    13,999  Mobile   
9  iQOO Z10x 5G (Ultramarine, 6GB RAM

In [25]:
# ============================================================================
# SAVE COMBINED DATASET
# ============================================================================

# Save the deduplicated combined dataset
combined_filename = 'combined_cleaned_data.csv'
df_combined_dedup.to_csv(combined_filename, index=False)

print(f"✅ Combined dataset saved to: {combined_filename}\n")

# Display breakdown by Type
print("📊 Product Type Distribution:")
type_counts = df_combined_dedup['Type'].value_counts()
print(type_counts)

# Display breakdown by Source
print(f"\n📊 Source Distribution:")
source_counts = df_combined_dedup['Source'].value_counts()
print(source_counts)

# Final verification for ML ready
print("\n" + "="*70)
print("🎯 COMBINED DATASET READY FOR TF-IDF + COSINE SIMILARITY")
print("="*70)
print(f"✓ Total products: {len(df_combined_dedup)}")
print(f"✓ All product names cleaned: {df_combined_dedup['Product_Name_Clean'].notna().all()}")
print(f"✓ All prices numeric: {df_combined_dedup['Price_Numeric'].dtype == 'float64'}")
print(f"✓ No missing values: {df_combined_dedup.isnull().sum().sum() == 0}")
print(f"✓ File: {combined_filename} (Ready to use in app.py)")
print("="*70)

✅ Combined dataset saved to: combined_cleaned_data.csv

📊 Product Type Distribution:
Type
Laptop             320
Shoes              240
Smartphone         232
Headphones         231
Television         202
Washing Machine    178
Refrigerator       162
Book               162
Jeans              152
Skincare           148
Jacket             146
Shirt              136
T-Shirt            129
Air Fryer          124
Tablet             116
Smartwatch         100
Camera              98
Pressure Cooker     95
Microwave           94
Vacuum Cleaner      88
Kurta               88
Water Purifier      84
Mobile              80
Toothbrush          72
Shaving             71
Dishwasher          70
Drone               62
Soap                59
Book Set            58
Epilator            52
Sunscreen           52
Lipstick            50
Body Wash           41
Name: count, dtype: int64

📊 Source Distribution:
Source
Flipkart    2003
Amazon      1989
Name: count, dtype: int64

🎯 COMBINED DATASET READY FOR TF-I

In [21]:
# Load both cleaned datasets
df_original = pd.read_csv('cleaned_data.csv')
df_bigdata = pd.read_csv('cleaned_bigdata.csv')

print("📂 Original Dataset (cleaned_data.csv):")
print(f"  • Shape: {df_original.shape}")
print(f"  • Columns: {df_original.columns.tolist()}\n")

print("📂 BigData Dataset (cleaned_bigdata.csv):")
print(f"  • Shape: {df_bigdata.shape}")
print(f"  • Columns: {df_bigdata.columns.tolist()}\n")

# Check for column consistency
print("✅ Columns match:", set(df_original.columns) == set(df_bigdata.columns))

📂 Original Dataset (cleaned_data.csv):
  • Shape: (176, 6)
  • Columns: ['Product Name', 'Price', 'Type', 'Source', 'Product_Name_Clean', 'Price_Numeric']

📂 BigData Dataset (cleaned_bigdata.csv):
  • Shape: (3818, 6)
  • Columns: ['Product_Name_Extracted', 'Price', 'Type', 'Source', 'Product_Name_Clean', 'Price_Numeric']

✅ Columns match: False


## Step 9: Combine Both Cleaned Datasets
Merge the original cleaned_data.csv with the new cleaned_bigdata.csv for a larger, more diverse product database